# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JasperOwen/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

import duckdb
from huggingface_hub import login

login(token=hf_token)

con = duckdb.connect()
con.sql("SET enable_http_metadata_cache=true;")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=hf_token
)

file_path

'/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet'

In [3]:
page_performance_query = f"""
SELECT
    client_hash_id,
    content_hash_id,

    -- Week 1 (March 1-6)
    SUM(CASE WHEN report_date <= '2026-03-07' THEN gsc_clicks ELSE 0 END) AS week1_clicks,
    SUM(CASE WHEN report_date <= '2026-03-07' THEN gsc_impressions ELSE 0 END) AS week1_impressions,
    AVG(CASE WHEN report_date <= '2026-03-07' AND gsc_avg_position > 0 THEN gsc_avg_position ELSE NULL END) AS week1_avg_position,

    -- Week 2 (March 8-15)
    SUM(CASE WHEN report_date > '2026-03-07' AND report_date <= '2026-03-15' THEN gsc_clicks ELSE 0 END) AS week2_clicks,
    SUM(CASE WHEN report_date > '2026-03-07' AND report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS week2_impressions,
    AVG(CASE WHEN report_date > '2026-03-07' AND report_date <= '2026-03-15' AND gsc_avg_position > 0 THEN gsc_avg_position ELSE NULL END) AS week2_avg_position,

FROM read_parquet('{file_path}')
WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
"""

page_performance = con.sql(page_performance_query).df()
page_performance.head()

,client_hash_id,content_hash_id,week1_clicks,week1_impressions,week1_avg_position,week2_clicks,week2_impressions,week2_avg_position
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,1.0,195.0,4.335620,1.0,184.0,3.774533
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,7.0,1437.0,3.991533,3.0,899.0,4.647973
2,client_65de48885f4ef01b,content_3c286ded8bd68120,4.0,410.0,8.486905,3.0,494.0,9.139858
3,client_65de48885f4ef01b,content_b2108e8fe3360fa6,1.0,280.0,6.158257,0.0,50.0,4.109989
4,client_65de48885f4ef01b,content_ff867882e604fa96,0.0,4.0,1.750000,0.0,20.0,3.950000


In [4]:
import numpy as np
import pandas as pd

def position_tier(pos):
    if pd.isna(pos) or pos == 0:
      return np.nan

    if pos <= 3:
        return 1
    elif pos <= 10:
        return 2
    elif pos <= 20:
        return 3
    elif pos <= 50:
        return 4
    else:
        return 5

page_performance["week1_position_tier"] = page_performance["week1_avg_position"].apply(position_tier)
page_performance["week2_position_tier"] = page_performance["week2_avg_position"].apply(position_tier)
page_performance["tier_change"] = page_performance["week2_position_tier"] - page_performance["week1_position_tier"]

page_performance[["client_hash_id", "content_hash_id", "week1_avg_position", "week2_avg_position",
        "week1_position_tier", "week2_position_tier", "tier_change"]].head(10)

,client_hash_id,content_hash_id,week1_avg_position,week2_avg_position,week1_position_tier,week2_position_tier,tier_change
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,4.335620,3.774533,2.0,2.0,0.0
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,3.991533,4.647973,2.0,2.0,0.0
2,client_65de48885f4ef01b,content_3c286ded8bd68120,8.486905,9.139858,2.0,2.0,0.0
3,client_65de48885f4ef01b,content_b2108e8fe3360fa6,6.158257,4.109989,2.0,2.0,0.0
4,client_65de48885f4ef01b,content_ff867882e604fa96,1.750000,3.950000,1.0,2.0,1.0
5,client_65de48885f4ef01b,content_d6c71358297cfd6a,8.557870,2.200000,2.0,1.0,-1.0
6,client_65de48885f4ef01b,content_6cdc3980c61ba7e9,6.538462,NaN,2.0,NaN,NaN
7,client_c182d11e4862a37d,content_a0e19c582cf792a7,6.211128,NaN,2.0,NaN,NaN
8,client_c182d11e4862a37d,content_d926564dfe83536b,24.066011,27.739213,4.0,4.0,0.0
9,client_c182d11e4862a37d,content_f4a0e5c90b283626,15.097646,NaN,3.0,NaN,NaN


In [5]:
print(page_performance[["tier_change"]].isna().sum())
print(len(page_performance))
print(f"tier_change missing: {34852/len(page_performance):.1%}")

tier_change    49593
dtype: int64
63856
tier_change missing: 54.6%


In [6]:
def position_change_bucket(change):
    if pd.isna(change):
        return np.nan
    elif change < 0:
        return "improved"
    elif change == 0:
        return "stable"
    else:
        return "declined"

page_performance["position_change"] = page_performance["tier_change"].apply(position_change_bucket)

In [7]:
position_signal_table = (
    page_performance[page_performance["position_change"].notna()]
    ["position_change"]
    .value_counts()
    .reset_index()
)

position_signal_table.columns = ["position_change", "n"]

position_signal_table

,position_change,n
0,stable,10036
1,improved,2300
2,declined,1927


In [8]:
def visibility_bucket(change):
    if pd.isna(change):
        return np.nan
    elif change <= 8:
        return "low"
    elif change > 8 and change <= 459:
        return "medium"
    else:
        return "high"

page_performance["visibility_bucket"] = page_performance["week2_impressions"].apply(visibility_bucket)

In [9]:
visibility_table = (
    page_performance[page_performance["visibility_bucket"].notna()]
    ["visibility_bucket"]
    .value_counts()
    .reset_index()
)

visibility_table.columns = ["Visibility bucket", "n"]

print(visibility_table)

  Visibility bucket      n
0               low  35905
1            medium  20004
2              high   7947


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Position change verdict: MIXED

A meaningful subset of the pages with active search data are declining in ranking (13%), indicating that ranking decline is a reasonable signal for prioritising pages for refresh review. However, the short window means that some of the tier changes could be due to rank turbulence rather than content decay. This is shown by 16% of the pages with active search data experiencing an increase in ranking.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
position_signal_table = (
    page_performance[page_performance["position_change"].notna()]
    ["position_change"]
    .value_counts()
    .reset_index()
)

position_signal_table.columns = ["position_change", "n"]

position_signal_table

,position_change,n
0,stable,10036
1,improved,2300
2,declined,1927


Visibility verdict: MIXED

By itself, the visibility of each page cannot tell us if a page is declining or not. However, when used alongside position change it can be used to help us decide which pages to prioritise for review, as pages with high visibility have greater search exposure, meaning they will benefit more from being reviewed for refresh.

In [11]:
visibility_table = (
    page_performance[page_performance["visibility_bucket"].notna()]
    ["visibility_bucket"]
    .value_counts()
    .reset_index()
)

visibility_table.columns = ["Visibility bucket", "n"]

print(visibility_table)

  Visibility bucket      n
0               low  35905
1            medium  20004
2              high   7947


Click change verdict: MIXED

A measurable amount of the pages experienced a decline in clicks between week 1 and week 2 (7.9%). This indicates that a meaningful number of pages require review for refresh. However, as most pages are listed as stable (69.1%) and some are listed as improved (22.9%), the click change alone cannot determine if a page's performance is declining, or if the page needs reviewing.

In [12]:
page_performance["click_change"] = (
    page_performance["week2_clicks"] -
    page_performance["week1_clicks"]
)

def click_change_bucket(change):
    if pd.isna(change):
        return np.nan
    elif change < 0:
        return "declined"
    elif change == 0:
        return "stable"
    else:
        return "improved"

page_performance["click_change_bucket"] = (page_performance["click_change"].apply(click_change_bucket))

click_signal_table = (
    page_performance[page_performance["click_change_bucket"].notna()]
    ["click_change_bucket"]
    .value_counts()
    .reset_index()
)

click_signal_table.columns = ["click_change", "n"]

click_signal_table

,click_change,n
0,stable,44170
1,improved,14618
2,declined,5068


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### CTR-vs-position verdict: CONFIRMED

In the dataset, as the position tier of the pages increases (indicating that the pages are further down the rankings) the click-through-rate (CTR) of the page's decreases. This is shown by the average CTR of tier 1 being 0.88% while the average CTR of tier 5 is 0.1%. This support's Flyrank's logic that a low CTR can be used to indicate low rankings of a page and vice versa.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
page_performance["week2_ctr"] = (
    page_performance["week2_clicks"] /
    page_performance["week2_impressions"].replace(0, np.nan)
)

# Group CTR by the existing position tiers
ctr_position_table = (
    page_performance
    .groupby("week2_position_tier", dropna=False)
    .agg(
        n=("week2_ctr", "count"),
        clicks=("week2_clicks", "sum"),
        impressions=("week2_impressions", "sum")
    )
    .reset_index()
)

# Calculate aggregate CTR for each position tier
ctr_position_table["ctr"] = (
    ctr_position_table["clicks"] /
    ctr_position_table["impressions"]
)

ctr_position_table

,week2_position_tier,n,clicks,impressions,ctr
0,1.0,3931,13982.0,1591211.0,0.008787
1,2.0,12572,41971.0,5694405.0,0.007371
2,3.0,6648,18239.0,3367437.0,0.005416
3,4.0,8334,17140.0,9097390.0,0.001884
4,5.0,584,135.0,137822.0,0.000980
5,NaN,461,93.0,1581.0,0.058824


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The results suggest that the tested signals can provide useful evidence when prioritising pages for review, but none of the signals should be used on their own to determine if a page should be reviewed for refresh, or the action to carry out on a page

Position change, visibility and click change can help identify pages which may be in decline. CTR can also be used with ranking position to do this. These signals are most useful as supporting evidence for review rather than as definitive indicators that a page requires a specific action.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.